In [100]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time

In [101]:
import random
import math

In [102]:
random.seed(42)

In [103]:
tests = ['vrp_16_3_1', 'vrp_26_8_1', 'vrp_51_5_1', 'vrp_101_10_1', 'vrp_200_16_1', 'vrp_421_41_1']
thresholds = [(387, 280), (1019, 630), (713, 540), (1193, 830), (3719, 1400), (2392, 2000)]

In [104]:
def load_data(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        args = lines[0].split()
        n = int(args[0])
        v = int(args[1])
        c = float(args[2])
        req = list()
        points = list()
        for i in range(n):
            d, x, y = list(map(float, lines[1 + i].split()))
            req.append(d)
            points.append((x, y))
            
        return n, v, c, req, points

In [105]:
def check_vrp(n, v, c, req, points, paths):
    used = [0] * n
    taken = [0] * v

    if len(paths) != v:
        return 1e18
    
    for wh, path in enumerate(paths):
        for i in path:
            taken[wh] += req[i]
            used[i] += 1

    for i in range(v):
        if taken[i] > c:
            return 1e18
            
    for i in range(1, n):
        if used[i] != 1:
            return 1e18
            
    res = 0
    for path in paths:
        if len(path) == 0:
            continue
        res += math.dist(points[0], points[path[0]])
        
        for i in range(len(path) - 1):
            res += math.dist(points[path[i]], points[path[i + 1]])
            
        res += math.dist(points[path[-1]], points[0])

    return res

In [106]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [107]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, v, c, req, points = load_data(test)
        start = time.time()
        
        if not use_file:
            paths = method(n, v, c, req, points)
        else:
            paths = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_vrp(n, v, c, req, points, paths)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Пока будем решать задачу в декомпозированном виде:
1. Найдем сопоставление каждого покупателя -- доставщику
2. Решим на доставщике TSP

Идейно то будем решать так, но я не хочу каждый раз при изменениях в сопоставлении честно оптимизировать TSP. Поэтому воспользуюсь простой идеей, что если я буду несильно менять цикл (добавлять/удалять точку), то и цикл наверно несильно поменяется.

А именно при добавлении точки я найду для нее оптимальный Insert, при удалении точки и вовсе цикл менять не буду.

В качестве жадного решения будем делать следующее: 
1. Поддерживаем циклы изначально каждая вершина цикл с 0.
2. Ищем два цикла, которые можно склеить и если это возможно, то склеиваем -- находим наилучшее склеивание.
3. Продолжаем процесс, пока улучшение уменьшает счет или циклов больше, чем v.

In [108]:
!g++ -O2 -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [109]:
def greedy_vrp(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [110]:
test_method(greedy_vrp, "greedy_vrp", True)

Checking greedy_vrp
Execution time: 0.1929 seconds
Target function vrp_16_3_1: 1e+18
Passed vrp_16_3_1: 0
Execution time: 0.0113 seconds
Target function vrp_26_8_1: 1e+18
Passed vrp_26_8_1: 0
Execution time: 0.0097 seconds
Target function vrp_51_5_1: 1e+18
Passed vrp_51_5_1: 0
Execution time: 0.0118 seconds
Target function vrp_101_10_1: 833.5085732089909
Passed vrp_101_10_1: 1
Execution time: 0.0169 seconds
Target function vrp_200_16_1: 1e+18
Passed vrp_200_16_1: 0
Execution time: 0.0623 seconds
Target function vrp_421_41_1: 1958.7462601014645
Passed vrp_421_41_1: 2
Score: 8


Интересно...

Как мы видим жадное решение получило отличный скор на последнем тесте, но на некоторых тестах выдает невалидные решения (не может склеивать циклы достаточное количество раз из - за превышения $\text{capacity}$ курьера).
Попробуем добавить локальную оптимизацию, где мы берем вершину из какого - то цикла и пытаемся ее добавить в другой цикл. Чтобы бороться с невалидными циклами добавим в стоимость решения превышение $\text{capacity}$, умноженное на $\lambda$.

Так как мы ввели понятие штрафа, то и жадное решение чуть улучшим, а именно если жадник не нашел feasible решение, то он продолжает решать ту же задачу, но циклы можно склеивать даже при превышении $\text{capacity}$, но с тем же штрафом.

In [111]:
!g++ -O2 -std=c++2a cpp_methods/local_opt.cpp -o tmp/local_opt

In [112]:
def local_opt_vrp(test_file): 
    os.system(f"./tmp/local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [113]:
test_method(local_opt_vrp, "local_opt_vrp", True)

Checking local_opt_vrp
Execution time: 0.1825 seconds
Target function vrp_16_3_1: 290.4187167384623
Passed vrp_16_3_1: 1
Execution time: 0.0140 seconds
Target function vrp_26_8_1: 1e+18
Passed vrp_26_8_1: 0
Execution time: 0.0117 seconds
Target function vrp_51_5_1: 585.2635459339223
Passed vrp_51_5_1: 1
Execution time: 0.0128 seconds
Target function vrp_101_10_1: 827.2746190907471
Passed vrp_101_10_1: 2
Execution time: 0.0298 seconds
Target function vrp_200_16_1: 1e+18
Passed vrp_200_16_1: 0
Execution time: 0.1649 seconds
Target function vrp_421_41_1: 1953.879818891265
Passed vrp_421_41_1: 2
Score: 16


Заимпрувили!

А теперь добавим локальную оптимизацию, где мы меняем местами две вершины из разных циклов. Применяем ее, если не нашлась операция прошлая.

In [114]:
!g++ -O2 -std=c++2a cpp_methods/tuned_local_opt.cpp -o tmp/tuned_local_opt

In [115]:
def tuned_local_opt_vrp(test_file): 
    os.system(f"./tmp/tuned_local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [116]:
test_method(tuned_local_opt_vrp, "tuned_local_opt", True)

Checking tuned_local_opt
Execution time: 0.1887 seconds
Target function vrp_16_3_1: 285.9566139053619
Passed vrp_16_3_1: 1
Execution time: 0.0149 seconds
Target function vrp_26_8_1: 607.6509456345309
Passed vrp_26_8_1: 2
Execution time: 0.0135 seconds
Target function vrp_51_5_1: 564.3255458794799
Passed vrp_51_5_1: 1
Execution time: 0.0144 seconds
Target function vrp_101_10_1: 825.5411704596885
Passed vrp_101_10_1: 2
Execution time: 0.0531 seconds
Target function vrp_200_16_1: 1458.1919573093974
Passed vrp_200_16_1: 1
Execution time: 0.3736 seconds
Target function vrp_421_41_1: 1945.3088284221412
Passed vrp_421_41_1: 2
Score: 24


УФФФФ

А теперь добавим стохастики, а именно будем циклы разрывать на циклы поменьше. Будем идти по циклу последовательно и с вероятностью 1/4 его разрывать на текущей вершине -- все что до этого не было разорвано будет образовывать новый цикл.
Для новых циклов будем применять жадный алгоритм из начала с последующими локальными оптимизациями.

In [117]:
!g++ -O2 -std=c++2a cpp_methods/stoch_local_opt.cpp -o tmp/stoch_local_opt

In [118]:
def stoch_local_opt_vrp(test_file): 
    os.system(f"./tmp/stoch_local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [119]:
test_method(stoch_local_opt_vrp, "stoch_local_opt", True)

Checking stoch_local_opt
Execution time: 6.1407 seconds
Target function vrp_16_3_1: 278.9849406063866
Passed vrp_16_3_1: 2
Execution time: 6.0045 seconds
Target function vrp_26_8_1: 607.6509456345308
Passed vrp_26_8_1: 2
Execution time: 6.0053 seconds
Target function vrp_51_5_1: 527.6748210713286
Passed vrp_51_5_1: 2
Execution time: 6.0097 seconds
Target function vrp_101_10_1: 825.5411704596886
Passed vrp_101_10_1: 2
Execution time: 6.0235 seconds
Target function vrp_200_16_1: 1354.638679337719
Passed vrp_200_16_1: 2
Execution time: 6.1712 seconds
Target function vrp_421_41_1: 1918.465241030167
Passed vrp_421_41_1: 2
Score: 30


Круто! Жадник с локальными оптимизациями показал себя мощно.

Хотя мы и прошли все тесты, я написал еще оптимальное решение, которое работает за $\mathcal{O}(3^N \cdot V)$